## This notebook is made to cover the practical part of the Machine Learning project that is related to creating a classification prediction model

Before introducing any kind of information in the dataset and/or features, the dataset **MUST BE SPLITTED**

Since we are dealing with a small unbalances dataset 4424 records, the split is going to be 80-20 with K-fold cross-validation to avoid problems with the target column since that is the feature we want to predict. The number of folds we do in the train data could also be changed to see which result we can get out of this parameter, first we are going to start with k = 10

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

raw_dataframe = pd.read_csv('dropout_dataset.csv', sep=';')


#Before really splitting the data, some cleaning is done to avoid annoying errors in the future

for column in raw_dataframe.columns:
    raw_dataframe.rename(columns = {f'{column}':f'{column.rstrip().lstrip()}'})



X = raw_dataframe.drop(columns=["Target"])
y = raw_dataframe["Target"]



X_train, X_test, Y_train, Y_test = train_test_split(X,y,test_size=0.2, random_state=42)


Now that the split have been done, we need to define a strategy to deal with the imbalanced aspect of the dataset. In the reference paper, they used SMOTE, ADASYN and Logistic regression, regarding the avaliable resampling methods, we can try to apply 
- Oversampling techniques
  1. Borderline-SMOTE
  2. random over-sampling
  3. SMOTE-ENN
  4. SMOTENC (especifically designed for nominal and continuous features)

OR

- Algorithmic Solutions
  1. Weighted loss function
  2. Classical weight adjustment
  3. Focal loss


Since our dataset is small, there will be a very big loss of information if we do under-sampling because reducing even more the dataset will pose even more problems, the oversampling techniques will be used for this case

Regarding the algorithms to build the models, since our task is to classificate or fit into classification the three categories of students based on the given existing features:
- Success
- Relative success
- Failure

And knowing that in the reference study they used **Logistic regression, SVM, decision tree, random forest, Gradient boosting, Xtreme gradient boosting, legit boost and cat boost**. Some other avaliable options for algorithms are:
- **Probabilistic & Linear Models**
    1. Naïve Bayes (NB)
- **Neural Networks**
    1. Multi-Layer Perceptron (MLP - Feedforward Neural Network)
- **Rule-Based & Distance-Based Models**
    1. K-Nearest Neighbors (KNN)
- **Ensemble & Evolutionary Methods**
    1. Voting classifier
    2. Genetic algorithms

**1. First part of the study: For each model (or one selected model), we can test different sample spaces -> 4 x N_models** 
This will give the output of which sampling method should be used in the study

**2. Second part of the study: for each model, which one can give more satisfactory results -> N_models**
This will give the output of which model is more appropriate for this task

**3. For the most appropriate model, what k-fold is ideal for getting better results(evaluate if it makes sense)**

**4. Compare the results with the study results and draw conclusions**

## Which sample strategy yields the best result?

To answer this question, we will take a similar approach to the reference paper, we pick first a classification method and then run that method with the different sample spaces that we have and then compare the different performances of them.
Taking into consideration our dataset carachteristics and the goal of the model (correctly classify the most under-represented class in the sample set "Partial success/enrolled"), the most appropriated ones are the ones that evaluates how well the classification is done to the most crititcal class, with that in mind, the most appropriate evaluation method is the F1 score (the same one used in the paper). This also keeps the comparision fair so we don't end up in the case of comparing apples to oranges.

The naive bayes method is the one that would require the biggest dataset treatment before applying the model and the KNN is one model that would suffer from the curse of dimensionality, since on this step, the goal is to evaluate which oversampling method is the best, we are going to proceed with the model that presents the most straight forward setup to this evaluation MLP.

To prepare our MLP model, these are the preparations that we need to do:
1. Normalize the continuous features
2. Normalize discrete numerical features
3. One-hot encode the categorical features
4. Normalize the ordinal features

#### 1.Normalize continuous features
To normalize them, we will use min-max scaling (we are aware that this causes loss of information)

In [2]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

X_train_treatment = X_train

df_continuous_columns = ['Previous qualification (grade)','Admission grade', 'Unemployment rate', 
    'Inflation rate','GDP', 'Curricular units 1st sem (grade)','Curricular units 2nd sem (grade)']

df_continuous_columns_data = X_train_treatment[df_continuous_columns]

df_continuous_columns_normalized = scaler.fit_transform(df_continuous_columns_data)

df_continuous_columns_normalized = pd.DataFrame(df_continuous_columns_normalized, columns=df_continuous_columns)

X_train_treatment[df_continuous_columns] = df_continuous_columns_normalized

#print(X_train_treatment[['Previous qualification (grade)', 'Admission grade']])

print("Dimension of the train dataset without the treatments = {}".format(X_train.shape))
print("Dimension of the train dataset with this treatment = {}".format(X_train_treatment.shape))

##With that, lets encapsulate the logic of this treatment

def normalize_continuous_columns(df):
    """
    Applies MinMaxScaler normalization to predefined continuous columns in the given dataset.

    Parameters:
        df (pd.DataFrame): The input dataset.

    Returns:
        pd.DataFrame: The dataset with normalized continuous columns.
    """
    scaler = MinMaxScaler()

    # Define the continuous columns (static list)
    df_continuous_columns = [
        'Previous qualification (grade)', 'Admission grade', 'Unemployment rate', 
        'Inflation rate', 'GDP', 'Curricular units 1st sem (grade)', 'Curricular units 2nd sem (grade)'
    ]

    # Select the data from these columns
    df_continuous_columns_data = df[df_continuous_columns]

    # Apply MinMaxScaler normalization
    df_continuous_columns_normalized = scaler.fit_transform(df_continuous_columns_data)

    # Convert back to DataFrame with the same column names
    df_continuous_columns_normalized = pd.DataFrame(df_continuous_columns_normalized, 
                                                    columns=df_continuous_columns, 
                                                    index=df.index)  # Ensure index is preserved

    # Replace original columns in the dataset
    df[df_continuous_columns] = df_continuous_columns_normalized

    return df

###Output check of the encapsulation


# X_train_treatment_encaps_1 = normalize_continuous_columns(X_train.copy())

# print(X_train_treatment_encaps_1[['Previous qualification (grade)', 'Admission grade']])  # Example check
# print("Dimension of the train dataset without the treatments = {}".format(X_train.shape))
# print("Dimension of the train dataset with this treatment = {}".format(X_train_treatment_encaps_1.shape))



Dimension of the train dataset without the treatments = (3539, 36)
Dimension of the train dataset with this treatment = (3539, 36)


As we can see from the output of the print statement, there is a some amount of NaN values, we will leave the treatment of those after we handle every column to avoid redundant work

#### 2.Normalize discrete numerical features
To normalize them, we will use min-max scaling (we are aware that this causes loss of information)

In [3]:
df_discrete_columns =['Age at enrollment', 'Curricular units 1st sem (credited)','Curricular units 1st sem (enrolled)',
     'Curricular units 1st sem (evaluations)','Curricular units 1st sem (approved)',
     'Curricular units 1st sem (without evaluations)', 'Curricular units 2nd sem (enrolled)',
     'Curricular units 2nd sem (evaluations)','Curricular units 2nd sem (approved)',
     'Curricular units 2nd sem (without evaluations)'
    ]

for column in df_discrete_columns:
    min_val = X_train_treatment[column].min()
    max_val = X_train_treatment[column].max()
    print(f"Column: {column}")
    print(f"Min: {min_val}, Max: {max_val}")
    print("-" * 50)

print("Dimension of the train dataset without the treatments = {}".format(X_train.shape))
print("Dimension of the train dataset with this treatment = {}".format(X_train_treatment.shape))

Column: Age at enrollment
Min: 17, Max: 70
--------------------------------------------------
Column: Curricular units 1st sem (credited)
Min: 0, Max: 20
--------------------------------------------------
Column: Curricular units 1st sem (enrolled)
Min: 0, Max: 26
--------------------------------------------------
Column: Curricular units 1st sem (evaluations)
Min: 0, Max: 45
--------------------------------------------------
Column: Curricular units 1st sem (approved)
Min: 0, Max: 26
--------------------------------------------------
Column: Curricular units 1st sem (without evaluations)
Min: 0, Max: 12
--------------------------------------------------
Column: Curricular units 2nd sem (enrolled)
Min: 0, Max: 23
--------------------------------------------------
Column: Curricular units 2nd sem (evaluations)
Min: 0, Max: 33
--------------------------------------------------
Column: Curricular units 2nd sem (approved)
Min: 0, Max: 20
--------------------------------------------------
C

Since our discrete columns have a fairly significant range, we will normalize them

In [4]:
discrete_scaler = MinMaxScaler()

df_discrete_columns_data = X_train_treatment[df_discrete_columns]

df_discrete_columns_normalized = discrete_scaler.fit_transform(df_discrete_columns_data)

df_discrete_columns_normalized = pd.DataFrame(df_discrete_columns_normalized, columns=df_discrete_columns)

X_train_treatment[df_discrete_columns] = df_discrete_columns_normalized

#print(X_train_treatment[['Age at enrollment', 'Curricular units 1st sem (credited)']])

print("Dimension of the train dataset without the treatments = {}".format(X_train.shape))
print("Dimension of the train dataset with this treatment = {}".format(X_train_treatment.shape))


def normalize_discrete_columns(df):
    """
    Applies MinMaxScaler normalization to predefined discrete columns in the given dataset.

    Parameters:
        df (pd.DataFrame): The input dataset.

    Returns:
        pd.DataFrame: The dataset with normalized discrete columns.
    """
    discrete_scaler = MinMaxScaler()

    # Define the discrete columns (static list)
    df_discrete_columns = [
        'Age at enrollment', 'Curricular units 1st sem (credited)', 'Curricular units 1st sem (enrolled)',
        'Curricular units 1st sem (evaluations)', 'Curricular units 1st sem (approved)',
        'Curricular units 1st sem (without evaluations)', 'Curricular units 2nd sem (enrolled)',
        'Curricular units 2nd sem (evaluations)', 'Curricular units 2nd sem (approved)',
        'Curricular units 2nd sem (without evaluations)'
    ]

    # Select the data from these columns
    df_discrete_columns_data = df[df_discrete_columns]

    # Apply MinMaxScaler normalization
    df_discrete_columns_normalized = discrete_scaler.fit_transform(df_discrete_columns_data)

    # Convert back to DataFrame with the same column names and index
    df_discrete_columns_normalized = pd.DataFrame(df_discrete_columns_normalized, 
                                                  columns=df_discrete_columns, 
                                                  index=df.index)  # Preserve index

    # Replace original columns in the dataset
    df[df_discrete_columns] = df_discrete_columns_normalized

    return df


# X_train_treatment_encaps_2 = normalize_discrete_columns(X_train.copy())

# print(X_train_treatment_encaps_2[['Age at enrollment', 'Curricular units 1st sem (credited)']])  # Example check
# print("Dimension of the train dataset without the treatments = {}".format(X_train.shape))
# print("Dimension of the train dataset with this treatment = {}".format(X_train_treatment_encaps_2.shape))



Dimension of the train dataset without the treatments = (3539, 36)
Dimension of the train dataset with this treatment = (3539, 36)


#### 3.One hot encode the categorical features

After normalizing the data in hand we can one hot encode all the categorical feature, this technique transform all the categories in a separate column (known as dummy column) where the values are binary, with that transformation, the data related to these columns is more digestable to the algorithm

In [5]:
from sklearn.preprocessing import OneHotEncoder

df_categorical_columns = ['Application mode', 'Course', 'Previous qualification', 'Nacionality',"Father's occupation", "Mother's occupation"]

encoder = OneHotEncoder(sparse_output=False, drop='first')

one_hot_encoded_categorical = encoder.fit_transform(X_train_treatment[df_categorical_columns])

# Fix: Ensure the new DataFrame retains the original index
encoded_df = pd.DataFrame(one_hot_encoded_categorical, 
                          columns=encoder.get_feature_names_out(df_categorical_columns),
                          ##Prevents pd to create rows to fit the new/unsetted index
                          ##If this is not explicitly done, pandas try to create new rows
                          ## to fit the mismatched indexes. This can introduce unwanted noise into our
                          ## sample space
                          index=X_train_treatment.index)


X_train_treatment = pd.concat([X_train_treatment.drop(columns=df_categorical_columns), encoded_df], axis=1)

print(X_train_treatment.head())
print("Dimension of the train dataset without the treatments = {}".format(X_train.shape))
print("Dimension of the train dataset with this treatment = {}".format(X_train_treatment.shape))


def one_hot_encode_categorical_columns(df):
    """
    Applies one-hot encoding to predefined categorical columns in the given dataset.
    
    Parameters:
        df (pd.DataFrame): The input dataset.

    Returns:
        pd.DataFrame: The dataset with one-hot encoded categorical columns.
    """
    # Define the categorical columns (static list)
    df_categorical_columns = [
        'Application mode', 'Course', 'Previous qualification', 
        'Nacionality', "Father's occupation", "Mother's occupation"
    ]

    # Initialize OneHotEncoder
    encoder = OneHotEncoder(sparse_output=False, drop='first')

    # Perform one-hot encoding
    one_hot_encoded_categorical = encoder.fit_transform(df[df_categorical_columns])

    # Convert back to DataFrame and maintain the original index
    encoded_df = pd.DataFrame(one_hot_encoded_categorical, 
                              columns=encoder.get_feature_names_out(df_categorical_columns), 
                              index=df.index)

    # Replace categorical columns with one-hot encoded data
    df = df.drop(columns=df_categorical_columns)
    df = pd.concat([df, encoded_df], axis=1)

    return df

# X_train_treatment_encaps_3 = one_hot_encode_categorical_columns(X_train.copy())

# print(X_train_treatment.head())
# print("Dimension of the train dataset without the treatments = {}".format(X_train.shape))
# print("Dimension of the train dataset with this treatment = {}".format(X_train_treatment_encaps_3.shape))


      Marital status  Application order  Daytime/evening attendance\t  \
3383               4                  1                             1   
2840               1                  1                             1   
564                1                  6                             1   
1786               2                  1                             1   
3900               1                  3                             1   

      Previous qualification (grade)  Mother's qualification  \
3383                        0.578947                      19   
2840                        0.494737                      37   
564                         0.401053                       1   
1786                        0.401053                      37   
3900                             NaN                      37   

      Father's qualification  Admission grade  Displaced  \
3383                       1         0.578947          0   
2840                      37         0.340000          0

#### 4. Normalize the ordinal features
Since for ordinal features the order is embedded on the distance between the values, we can just apply the regular normalization to avoid dimensionality problems with them

**On this case, we are considering that the loss of information in treating ordinal as categorical can prejudice the model somehow, this can be also another test to make**

Taking a closer look on the values of the preassumed ordinal features, we can see that the numbers **do not** hold any order or meaning e.g: 
- Can't read or write is 35, but Higher Education - Bachelor's Degree is 2, which is completely out of order.
- 12th Year of Schooling - Not Completed is 9, but 7th Year (Old) is 11, which doesn’t make sense in an ascending order.
- Can't read or write is 35, but Higher Education - Doctorate (3rd cycle) is 44.
- Basic education 1st cycle (4th/5th year) is 37, while Higher Education - Bachelor's Degree is 2.

With this observation we can choose two paths, treat them as categorical and one hot encode them or create groups to logically order them, the first path use the data as is and does not give any more meaning to it but the second one can enrich the data but the impact on the quality of the classification is unkown.

Since in general, scholarity level is considered ordinal, that is the approach we are going to take on this model

In [6]:
print(X_train_treatment["Father's qualification"].unique())
print(X_train_treatment["Mother's qualification"].unique())

[ 1 37 38 19  3 12 34  4  2  5 11 22 25 40 39 20 31 10 42 30 29 14 36  9
 41 43 26 27 44 13 33 35  6]
[19 37  1 38  2 34  3 12  4  5 41  9 39  6 40 42 11 43 10 29 30 36 27 14
 22 26 35]


In [7]:
macro_groups_of_higher_education = {
    'No formal Education': 1,
    'Primary Education (Basic Education - 1st Cycle)': 2,
    'Lower Secondary Education (Basic Education - 2nd Cycle)':3,
    'Upper Secondary Education (Basic Education - 3rd Cycle)':4,
    'Technical & Professional Education': 5,
    'Higher Education (University Level)':6,
    'Unknown':7
}

education_level_mapping_father_qualification = {
    35: 1,
    36: 1,
    37: 2,
    38: 3,
    26: 3,
    11: 3,
    30: 3,
    19: 4,
    29: 4,
    14: 4,
    12: 4,
    10: 4,
    9: 4,
    1: 4,
    22: 5,
    31: 5,
    18: 5,
    33: 5,
    25: 5,
    27: 5,
    20: 5,
    13: 5,
    39: 5,
    6: 6,
    2: 6,
    3: 6,
    40: 6,
    41: 6,
    42: 6,
    4: 6,
    43: 6,
    5: 6,
    44: 6,
    34: 7
}

##There is a difference on the qualification mapping

education_level_mapping_mother_qualification = {
    1: 4,
    2: 6,
    3: 6,
    4: 6,
    5: 6,
    6 :6,
    9: 4,
    10: 4,
    11: 3,
    12: 4,
    14: 4,
    18: 5,
    19: 4,
    22: 5,
    26: 3,
    27: 5,
    29: 4,
    30: 3,
    34: 7,
    35: 1,
    36: 1,
    37: 2,
    38: 3,
    39: 5,
    40: 6,
    41: 6,
    42: 6,
    43: 6,
    44: 6,
}

X_train_treatment["Father's qualification"] = X_train_treatment["Father's qualification"].map(education_level_mapping_father_qualification)

X_train_treatment["Mother's qualification"] = X_train_treatment["Mother's qualification"].map(education_level_mapping_father_qualification)


After logically mapping them, we can finally normalize the values

In [8]:
ordinal_scaler = MinMaxScaler()

df_ordinal_columns = ['Marital status',  'Application order', "Mother's qualification", "Father's qualification"]

df_ordinal_columns_data = X_train_treatment[df_ordinal_columns]

df_ordinal_columns_normalized = ordinal_scaler.fit_transform(df_ordinal_columns_data)

df_ordinal_columns_normalized = pd.DataFrame(df_ordinal_columns_normalized, columns=df_ordinal_columns, index=X_train_treatment.index)

X_train_treatment[df_ordinal_columns] = df_ordinal_columns_normalized

#print(X_train_treatment[['Marital status', 'Application order', "Mother's qualification", "Father's qualification"]])

def map_and_normalize_ordinal_columns(df):
    """
    Maps categorical education qualifications to numerical macro groups and 
    normalizes specified ordinal columns using MinMaxScaler.

    macro_groups_of_higher_education = {
        'No formal Education': 1,
        'Primary Education (Basic Education - 1st Cycle)': 2,
        'Lower Secondary Education (Basic Education - 2nd Cycle)':3,
        'Upper Secondary Education (Basic Education - 3rd Cycle)':4,
        'Technical & Professional Education': 5,
        'Higher Education (University Level)':6,
        'Unknown':7
    }

    Parameters:
        df (pd.DataFrame): The input dataset.

    Returns:
        pd.DataFrame: The dataset with mapped and normalized ordinal columns.
    """
    # Define the mapping dictionaries
    education_level_mapping_father_qualifications = {
        35: 1, 36: 1, 37: 2, 38: 3, 26: 3, 11: 3, 30: 3, 19: 4, 29: 4,
        14: 4, 12: 4, 10: 4, 9: 4, 1: 4, 22: 5, 31: 5, 18: 5, 33: 5,
        25: 5, 27: 5, 20: 5, 13: 5, 39: 5, 6: 6, 2: 6, 3: 6, 40: 6,
        41: 6, 42: 6, 4: 6, 43: 6, 5: 6, 44: 6, 34: 7
    }

    education_level_mapping_mother_qualifications = {
        1: 4, 2: 6, 3: 6, 4: 6, 5: 6, 6: 6, 9: 4, 10: 4, 11: 3, 12: 4,
        14: 4, 18: 5, 19: 4, 22: 5, 26: 3, 27: 5, 29: 4, 30: 3, 34: 7,
        35: 1, 36: 1, 37: 2, 38: 3, 39: 5, 40: 6, 41: 6, 42: 6, 43: 6, 44: 6
    }

    # Apply mappings to the dataset
    df["Father's qualification"] = df["Father's qualification"].map(education_level_mapping_father_qualifications)
    df["Mother's qualification"] = df["Mother's qualification"].map(education_level_mapping_mother_qualifications)

    # Define ordinal columns for normalization
    df_ordinal_columns = ['Marital status', 'Application order', "Mother's qualification", "Father's qualification"]

    # Normalize ordinal columns
    ordinal_scaler = MinMaxScaler()
    df_ordinal_columns_data = df[df_ordinal_columns]

    df_ordinal_columns_normalized = ordinal_scaler.fit_transform(df_ordinal_columns_data)
    df_ordinal_columns_normalized = pd.DataFrame(df_ordinal_columns_normalized, columns=df_ordinal_columns, index=df.index)

    # Replace original ordinal columns with normalized values
    df[df_ordinal_columns] = df_ordinal_columns_normalized

    return df


Now that our dataset is treated and ready to be ingested by the model we can proceed to the next steps. Just for comparision lets check the difference in dimension for the treated datased and the original one

In [9]:
print("Dimension of the train dataset without the treatments = {}".format(X_train.shape))
print("Dimension of the train dataset with the treatments = {}".format(X_train_treatment.shape))

#X_train_treatment_encaps_4 = map_and_normalize_ordinal_columns(X_train.copy())

#print(X_train_treatment_encaps_4[['Marital status', 'Application order', "Mother's qualification", "Father's qualification"]])


Dimension of the train dataset without the treatments = (3539, 36)
Dimension of the train dataset with the treatments = (3539, 172)


In [10]:
print("Xshape = {}".format(X_train_treatment.shape))
print("Yshape = {}".format(Y_train.shape))


merged_x_and_y = pd.concat([X_train_treatment, Y_train], axis=1)
print("Dimension of the train dataset with the treatments = {}".format(merged_x_and_y.shape))

merged_x_and_y = merged_x_and_y.dropna()

print("Dimension of the train dataset with the treatments = {}".format(merged_x_and_y.shape))

X_train_cleaned = merged_x_and_y.drop(columns=['Target'])
Y_train_cleaned = merged_x_and_y['Target']

print("X_new_test = {}".format(X_train_cleaned.shape))
print("Y_new_test = {}".format(Y_train_cleaned.shape))


#print(X_train_treatment_merged['Target'])


#print(X_train_treatment.isnull().any())

#print(Y_train.isnull().any())


#clf = MLPClassifier(random_state=1, max_iter=1000).fit(X_train_cleaned, Y_train_cleaned)

Xshape = (3539, 172)
Yshape = (3539,)
Dimension of the train dataset with the treatments = (3539, 173)
Dimension of the train dataset with the treatments = (2828, 173)
X_new_test = (2828, 172)
Y_new_test = (2828,)


### Comparing how well the MLP perform under different oversample techniques

### Create a sample space using borderline SMOTE

In [11]:
from collections import Counter
from imblearn.over_sampling import BorderlineSMOTE

print('Original training dataset shape %s' % Counter(Y_train_cleaned))

sm = BorderlineSMOTE(random_state=42)
X_borderline_SMOTE, Y_borderline_SMOTE = sm.fit_resample(X_train_cleaned,Y_train_cleaned)

print('Resampled training dataset shape %s' % Counter(Y_borderline_SMOTE))


Original training dataset shape Counter({'Graduate': 1445, 'Dropout': 868, 'Enrolled': 515})
Resampled training dataset shape Counter({'Dropout': 1445, 'Enrolled': 1445, 'Graduate': 1445})


### Create a sample space using random over sampler

In [12]:
from imblearn.over_sampling import RandomOverSampler

print('Original training dataset shape %s' % Counter(Y_train_cleaned))

ros = RandomOverSampler(random_state=42)
X_random_over_sampler, Y_random_over_sampler = ros.fit_resample(X_train_cleaned,Y_train_cleaned)

print('Resampled training dataset shape %s' % Counter(Y_random_over_sampler))


Original training dataset shape Counter({'Graduate': 1445, 'Dropout': 868, 'Enrolled': 515})
Resampled training dataset shape Counter({'Dropout': 1445, 'Enrolled': 1445, 'Graduate': 1445})


### Create a sample space using random over SMOTE EEN

Here, the result is a little different than those we saw in the other resample stretegies

In [13]:
from imblearn.combine import SMOTEENN

print('Original training dataset shape %s' % Counter(Y_train_cleaned))

sme = SMOTEENN(random_state=42)
X_smote_een, Y_smote_een = sme.fit_resample(X_train_cleaned,Y_train_cleaned)

print('Resampled training dataset shape %s' % Counter(Y_smote_een))


Original training dataset shape Counter({'Graduate': 1445, 'Dropout': 868, 'Enrolled': 515})
Resampled training dataset shape Counter({'Enrolled': 1098, 'Dropout': 736, 'Graduate': 355})


**The difference in the result set might make this method unsuitabel, even though we managed to increase the number of enrolled targets we significantly decreased the other ones, with that, we just shifted the imbalance of the data to the other targets. To confirm this hypothesis we need to run the tests**

### Create a sample space using random over SMOTENC

Here, the result is a little different than those we saw in the other resample stretegies

In [14]:
from imblearn.over_sampling import SMOTENC

print('Original training dataset shape %s' % Counter(Y_train_cleaned))

smnc = SMOTENC(random_state=42, categorical_features= [1,3,5,7])
X_smote_smnc, Y_smote_smnc = smnc.fit_resample(X_train_cleaned,Y_train_cleaned)

print('Resampled training dataset shape %s' % Counter(Y_smote_smnc))

Original training dataset shape Counter({'Graduate': 1445, 'Dropout': 868, 'Enrolled': 515})
Resampled training dataset shape Counter({'Dropout': 1445, 'Enrolled': 1445, 'Graduate': 1445})


#### Applying the same preprocessing to the test data

Before actually comparing between those, we need to apply the preprocessing we did on the train data to the test data

### Now that the samplespaces are created, we need to create a model to using each one of those and evaluate their performance to select one of the methods

In [15]:
# Copy the original dataset to avoid modifying the raw data
X_test_treated = X_test.copy()

# Step 1: Normalize discrete numerical columns
X_test_treated = normalize_discrete_columns(X_test_treated)

# Step 2: One-Hot Encode categorical columns
X_test_treated = one_hot_encode_categorical_columns(X_test_treated)

# Step 3: Map and normalize ordinal columns
X_test_treated = map_and_normalize_ordinal_columns(X_test_treated)

# Print results and check dimensions
print(X_test_treated.head())
print("Dimension of the test dataset without the treatments = {}".format(X_test.shape))
print("Dimension of the test dataset with the treatments = {}".format(X_test_treated.shape))

      Marital status  Application order  Daytime/evening attendance\t  \
1255             0.6                0.0                             1   
3458             0.0                0.0                             1   
3390             0.0                0.0                             1   
1497             0.0                0.2                             1   
1536             0.0                0.0                             1   

      Previous qualification (grade)  Mother's qualification  \
1255                           133.1                0.833333   
3458                           125.0                0.833333   
3390                           133.0                0.333333   
1497                           110.0                0.500000   
1536                           130.0                0.166667   

      Father's qualification  Admission grade  Displaced  \
1255                0.500000            110.0          1   
3458                0.833333            119.8          0

In [16]:
from sklearn.neural_network import MLPClassifier


#borderline_SMOTE_MLP = MLPClassifier(random_state=1, max_iter=1000).fit(X_borderline_SMOTE, Y_borderline_SMOTE)
#random_over_sampler_MLP = MLPClassifier(random_state=1, max_iter=1000).fit(X_random_over_sampler, Y_random_over_sampler)
#SMOTE_EEN_MLP = MLPClassifier(random_state=1, max_iter=1000).fit(X_smote_een, Y_smote_een)
#SMOTE_NC_MLP = MLPClassifier(random_state=1, max_iter=1000).fit(X_smote_smnc, Y_smote_smnc)

In [17]:
#Y_pred_borderline_SMOTE_MLP = borderline_SMOTE_MLP.predict()